# OLS Adapter Tutorial

[OLS](https://www.ebi.ac.uk/ols4/) (the Ontology Lookup Service) is EBI's ontology
repository and search engine. It serves hundreds of ontologies through a REST API, always
at their latest release, with nothing to download.

The **OLS adapter** puts that API behind the standard OAK interfaces, so the same code that
works against a local file works against OLS. That makes it a good fit when you want to:

- look terms up in an ontology you have not downloaded (and do not want to)
- search across *every* ontology EBI indexes at once
- get labels, definitions, synonyms and cross-references that are always current
- work with ontologies too large to load locally

The trade-off is latency: every lookup is an HTTP request. This notebook shows what the
adapter can do and how to keep the request count down.

## Connecting

The selector is `ols:<ontology>`; a bare `ols:` searches across all ontologies. There is
also `tib:` for the [TIB](https://service.tib.eu/ts4tib/) OLS instance.

In [1]:
import logging
import warnings

logging.getLogger("oaklib").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
%env PYTHONWARNINGS=ignore

from oaklib import get_adapter

adapter = get_adapter("ols:go")
print(type(adapter).__name__, "|", adapter.focus_ontology)

env: PYTHONWARNINGS=ignore


OlsImplementation | go


Ontology-level metadata tells you exactly which release you are querying -- worth recording
if you are producing results you need to reproduce later:

In [2]:
for predicate, values in adapter.ontology_metadata_map("go").items():
    print(f"{predicate:22s} {values[0]}")

dcterms:title          Gene Ontology
dcterms:description    The Gene Ontology (GO) provides a framework and set of concepts for describing the functions of gene products from all organisms.
owl:versionInfo        2026-06-15
owl:versionIRI         http://purl.obolibrary.org/obo/go/releases/2026-06-15/extensions/go-plus.ofn
schema:url             http://purl.obolibrary.org/obo/go/extensions/go-plus.owl
foaf:homepage          http://geneontology.org/


## Basic lookups

Labels and definitions work exactly as they do for a local file:

In [3]:
NUCLEUS = "GO:0005634"

print(adapter.label(NUCLEUS))
print()
print(adapter.definition(NUCLEUS))

nucleus

A membrane-bounded organelle of eukaryotic cells in which chromosomes are housed and replicated. In most cells, the nucleus contains all of the cell's chromosomes except the organellar chromosomes, and is the site of RNA synthesis and processing. In some species, or in specialized cell types, RNA metabolism or DNA replication may be absent.


Definitions carry their provenance, which OLS reports alongside the definition text:

In [4]:
for curie, definition, metadata in adapter.definitions([NUCLEUS], include_metadata=True):
    print(f"{curie}: {definition[:60]}...")
    print(f"  provenance: {metadata}")

GO:0005634: A membrane-bounded organelle of eukaryotic cells in which ch...
  provenance: {'oio:hasDbXref': ['GOC:curators']}


### One term, one request

A term lookup returns everything OLS knows about that term, so the adapter caches the whole
payload. The first call to *any* accessor fetches the term; every other accessor is then
served from the cache. This matters: naively, the block below would be five HTTP round
trips.

In [5]:
import time

fresh = get_adapter("ols:go")

start = time.time()
fresh.label(NUCLEUS)
first_call = time.time() - start

start = time.time()
fresh.definition(NUCLEUS)
fresh.entity_aliases(NUCLEUS)
fresh.entity_metadata_map(NUCLEUS)
list(fresh.terms_subsets([NUCLEUS]))
remaining_calls = time.time() - start

print(f"first accessor (fetches the term):   {first_call:.3f}s")
print(f"four more accessors (served cached): {remaining_calls:.3f}s")

first accessor (fetches the term):   0.838s
four more accessors (served cached): 0.000s


## Synonyms, with scope and provenance

OLS records the scope of each synonym, the publications supporting it, and any
ontology-specific synonym type:

In [6]:
print(sorted(adapter.entity_aliases(NUCLEUS)))
print()
for predicate, values in sorted(adapter.entity_alias_map(NUCLEUS).items()):
    print(f"{predicate:22s} {values}")

['cell nucleus', 'horsetail nucleus', 'nucleus']

oio:hasExactSynonym    ['cell nucleus']
oio:hasNarrowSynonym   ['horsetail nucleus']
rdfs:label             ['nucleus']


In [7]:
for _, synonym in sorted(adapter.synonym_property_values([NUCLEUS]), key=lambda s: s[1].val):
    print(f"{synonym.pred:18s} {synonym.val!r}")
    print(f"  xrefs: {sorted(synonym.xrefs)}")

hasExactSynonym    'cell nucleus'
  xrefs: []
hasNarrowSynonym   'horsetail nucleus'
  xrefs: ['GOC:al', 'GOC:mah', 'GOC:vw', 'PMID:15030757']


## Metadata, cross-references and subsets

`entity_metadata_map` translates the OLS record into the predicates OAK uses elsewhere, so
downstream code does not need to know it came from a web service:

In [8]:
for predicate, values in sorted(adapter.entity_metadata_map(NUCLEUS).items()):
    rendered = str(sorted(values))
    print(f"{predicate:22s} {rendered[:90]}")

IAO:0000115            ["A membrane-bounded organelle of eukaryotic cells in which chromosomes are housed and rep
oio:hasDbXref          ['NIF_Subcellular:sao1702920020', 'Wikipedia:Cell_nucleus']
oio:hasExactSynonym    ['cell nucleus']
oio:hasNarrowSynonym   ['horsetail nucleus']
oio:hasOBONamespace    ['cellular_component']
oio:id                 ['GO:0005634']
oio:inSubset           ['goslim_agr', 'goslim_candida', 'goslim_chembl', 'goslim_drosophila', 'goslim_euk_cellula
rdfs:label             ['nucleus']


In [9]:
print("cross-references:")
for predicate, object_id in adapter.simple_mappings_by_curie(NUCLEUS):
    print(f"  {predicate} {object_id}")

print()
print("subsets:")
print(" ", sorted(subset for _, subset in adapter.terms_subsets([NUCLEUS])))

cross-references:
  oio:hasDbXref NIF_Subcellular:sao1702920020
  oio:hasDbXref Wikipedia:Cell_nucleus

subsets:
  ['goslim_agr', 'goslim_candida', 'goslim_chembl', 'goslim_drosophila', 'goslim_euk_cellular_processes_ribbon', 'goslim_flybase_ribbon', 'goslim_generic', 'goslim_metagenomics', 'goslim_mouse', 'goslim_pir', 'goslim_plant', 'goslim_plant_ribbon', 'goslim_yeast']


Terms can also be projected into [OBO Graph](https://github.com/geneontology/obographs)
nodes, the same structure the file-based adapters produce:

In [10]:
node = adapter.node(NUCLEUS, include_metadata=True)
print(node.id, "|", node.lbl, "|", node.type)
print("definition xrefs:", node.meta.definition.xrefs)
print("xrefs:           ", [xref.val for xref in node.meta.xrefs])
print("synonyms:        ", sorted((s.pred, s.val) for s in node.meta.synonyms))
print("deprecated:      ", node.meta.deprecated)

GO:0005634 | nucleus | CLASS
definition xrefs: ['GOC:curators']
xrefs:            ['NIF_Subcellular:sao1702920020', 'Wikipedia:Cell_nucleus']
synonyms:         [('hasExactSynonym', 'cell nucleus'), ('hasNarrowSynonym', 'horsetail nucleus')]
deprecated:       None


Obsolete terms report their replacement, so OLS can be used to migrate stale identifiers:

In [11]:
OBSOLETE = "GO:0000004"

metadata = adapter.entity_metadata_map(OBSOLETE)
print("deprecated:  ", metadata.get("owl:deprecated"))
print("replaced by: ", metadata.get("IAO:0100001"))
for _, predicate, replacement in adapter.obsoletes_migration_relationships([OBSOLETE]):
    print(f"  {OBSOLETE} --{predicate}--> {replacement} ! {adapter.label(replacement)}")

deprecated:   [True]
replaced by:  ['GO:0008150']


  GO:0000004 --IAO:0100001--> GO:0008150 ! biological_process


## Search

OLS search is the adapter's fastest operation, since it is a single indexed request. With a
focus ontology results are restricted to it:

In [12]:
import itertools

for curie in itertools.islice(adapter.basic_search("nuclear membrane"), 5):
    print(f"{curie:15s} {adapter.label(curie)}")

GO:0031965      nuclear membrane
GO:0046745      viral capsid secondary envelopment
GO:0046765      viral budding from nuclear membrane
GO:1990919      proteasome-nuclear membrane anchor activity
GO:0046771      viral budding from inner nuclear membrane


Without a focus ontology you are searching everything OLS indexes at once -- useful when you
do not yet know which ontology a term lives in:

In [13]:
everything = get_adapter("ols:")

for curie in itertools.islice(everything.basic_search("Parkinson disease"), 8):
    print(f"{curie:20s} {everything.label_cache.get(curie)}")

http://id.nlm.nih.gov/mesh/D010300 Parkinson Disease
NCIT:C26845          Parkinson Disease
MONDO:0005180        Parkinson disease
OMIT:0011296         Parkinson Disease
http://id.nlm.nih.gov/mesh/D020734 Parkinsonian Disorders
MONDO:0010820        autosomal recessive juvenile Parkinson disease 2
MONDO:0008199        late-onset Parkinson disease
MONDO:0011764        autosomal dominant Parkinson disease 8


Search can be restricted to particular fields, and to exact matches:

In [14]:
from oaklib.datamodels.search import SearchConfiguration, SearchProperty

config = SearchConfiguration(properties=[SearchProperty.LABEL], is_complete=True, limit=5)
for curie in adapter.basic_search("vacuole", config):
    print(f"{curie:15s} {adapter.label(curie)}")

GO:0005773      vacuole


## Graph traversal

Ancestors and descendants are served by dedicated OLS endpoints, which return the full
transitive closure in one paged query -- much cheaper than walking the hierarchy edge by
edge.

OLS distinguishes the `is_a` closure from the "hierarchical" closure (`is_a` plus the
ontology's configured part-of style relations), and the adapter picks the endpoint from the
predicates you pass:

In [15]:
from oaklib.datamodels.vocabulary import IS_A, PART_OF

VACUOLE = "GO:0005773"

isa_only = set(adapter.ancestors(VACUOLE, predicates=[IS_A], reflexive=False))
hierarchical = set(adapter.ancestors(VACUOLE, predicates=[IS_A, PART_OF], reflexive=False))

print(f"is_a ancestors:           {len(isa_only)}")
print(f"is_a + part_of ancestors: {len(hierarchical)}")
print()
print("reachable only via part_of:")
for curie in sorted(hierarchical - isa_only):
    print(f"  {curie} ! {adapter.label(curie)}")

is_a ancestors:           12
is_a + part_of ancestors: 15

reachable only via part_of:


  CL:0000000 ! cell


  GO:0005622 ! intracellular anatomical structure


  GO:0005737 ! cytoplasm


Descendant queries are paged all the way through, so counts for high-level terms are
complete rather than truncated at the first page. The gap between the two closures is
large: most things inside the nucleus are *part of* it rather than *kinds of* it.

In [16]:
isa_descendants = list(adapter.descendants(NUCLEUS, predicates=[IS_A]))
hierarchical_descendants = list(adapter.descendants(NUCLEUS, predicates=[IS_A, PART_OF]))
cc_descendants = list(adapter.descendants("GO:0005575", predicates=[IS_A, PART_OF]))

print(f"is_a descendants of nucleus:           {len(isa_descendants)}")
print(f"is_a + part_of descendants of nucleus: {len(hierarchical_descendants)}")
print(f"descendants of cellular_component:     {len(cc_descendants)}")

is_a descendants of nucleus:           25
is_a + part_of descendants of nucleus: 474
descendants of cellular_component:     4087


Direct relationships come from the OLS term graph endpoint:

In [17]:
NUCLEAR_MEMBRANE = "GO:0031965"

for s, p, o in sorted(adapter.relationships(subjects=[NUCLEAR_MEMBRANE])):
    print(f"{s} ! {adapter.label(s)}  --{adapter.label(p) or p}-->  {o} ! {adapter.label(o)}")

GO:0031965 ! nuclear membrane  --BFO:0000050-->  GO:0005635 ! nuclear envelope


GO:0031965 ! nuclear membrane  --RO:0002162-->  NCBITaxon:2759 ! Eukaryota


GO:0031965 ! nuclear membrane  --rdfs:subClassOf-->  GO:0031090 ! organelle membrane


## Multilingual ontologies

OLS holds translations for some ontologies. `languages()` reports which are available, and
`label` takes a language tag:

In [18]:
hpo = get_adapter("ols:hp")

print("languages:", sorted(hpo.languages()))
print("multilingual:", hpo.multilingual)
print()
PHENOTYPIC_ABNORMALITY = "HP:0000118"
for lang in ["en", "fr", "nl", "cs"]:
    print(f"  {lang}: {hpo.label(PHENOTYPIC_ABNORMALITY, lang=lang)}")

languages: ['cs', 'de', 'dtp', 'en', 'es', 'fr', 'it', 'ja', 'nl', 'nna', 'pt', 'tr', 'tw', 'zh']


multilingual: True



  en: Phenotypic abnormality


  fr: Anomalie phénotypique


  nl: Fenotypische abnormaliteit


  cs: Fenotypová abnormalita


## Which ontologies are available?

With no focus ontology, `ontologies()` lists everything the OLS instance serves:

In [19]:
all_ontologies = sorted(everything.ontologies())
print(f"{len(all_ontologies)} ontologies available")
print(all_ontologies[:20])

282 ontologies available
['addicto', 'ado', 'aeo', 'afo', 'afpo', 'agro', 'aism', 'amphx', 'apo', 'apollo_sv', 'aro', 'bao', 'bcgo', 'bcio', 'bco', 'bfo', 'biolink', 'bmont', 'bspo', 'bto']


## Performance: what is cheap and what is not

Every operation is an HTTP request, so the shape of your code matters far more here than it
does with a local file. Roughly:

| operation | cost |
| --- | --- |
| `basic_search` | one request |
| any accessor on a term (label, definition, synonyms, xrefs, metadata) | one request for the *first* accessor, then free |
| `ancestors` / `descendants` | one request per page of results |
| `entities()` | one request per 500 terms -- avoid on large ontologies |

The rule of thumb: work term-by-term rather than property-by-property, and let the cache do
the rest.

In [20]:
terms = ["GO:0005634", "GO:0005773", "GO:0005737", "GO:0016020", "GO:0031965"]

fresh = get_adapter("ols:go")
start = time.time()
rows = [
    (curie, fresh.label(curie), len(fresh.entity_aliases(curie)))
    for curie in terms
]
elapsed = time.time() - start

for curie, label, n_aliases in rows:
    print(f"{curie:12s} {label:20s} {n_aliases} aliases")
print(f"\n{len(terms)} terms, label + aliases each: {elapsed:.2f}s")

GO:0005634   nucleus              3 aliases
GO:0005773   vacuole              2 aliases
GO:0005737   cytoplasm            1 aliases
GO:0016020   membrane             7 aliases
GO:0031965   nuclear membrane     1 aliases

5 terms, label + aliases each: 4.09s


If you need to make many queries against the same ontology, consider downloading it once and
using a local adapter instead -- OAK's interfaces are the same either way, so only the
selector changes:

```python
adapter = get_adapter("ols:go")            # remote, always current, one request per term
adapter = get_adapter("sqlite:obo:go")     # local, downloaded once, sub-millisecond lookups
```

## Command line

The same selector works from the command line:

In [21]:
!runoak -i ols:go --quiet info GO:0005634

GO:0005634 ! nucleus


In [22]:
!runoak -i ols:go --quiet search 'nuclear membrane' | head -5

GO:0031965 ! nuclear membrane
GO:0046745 ! viral capsid secondary envelopment
GO:0046765 ! viral budding from nuclear membrane
GO:1990919 ! proteasome-nuclear membrane anchor activity
GO:0046771 ! viral budding from inner nuclear membrane


## Further reading

- [OLS adapter API documentation](https://incatools.github.io/ontology-access-kit/implementations/ols.html)
- [OLS4 API documentation](https://www.ebi.ac.uk/ols4/help)
- [Selectors](https://incatools.github.io/ontology-access-kit/packages/selectors.html)